# Lab 5 Agent Loop
## MM my own implementation

In [1]:
# imports
from openai import OpenAI
from dotenv import load_dotenv
import json

load_dotenv(override=True)
client = OpenAI()

In [2]:
# helper fns
todos = []

def get_todo_report() -> str:
    lines = []
    for i, todo in enumerate(todos, 1):
        mark = "DONE" if todo["done"] else "   "
        lines.append(f"#{i}: [{mark}] {todo['description']}")
    return "\n".join(lines)

def create_todos(descriptions: list[str]) -> str:
    for desc in descriptions:
        todos.append({"description": desc, "done": False})
    return get_todo_report()

def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        todos[index - 1]["done"] = True
        print(completion_notes)
    else:
        return "Invalid index."
    return get_todo_report()

In [3]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "create_todos",
            "description": "Add new todos and return the full list",
            "parameters": {
                "type": "object",
                "properties": {
                    "descriptions": {"type": "array", "items": {"type": "string"}}
                },
                "required": ["descriptions"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "mark_complete",
            "description": "Mark the todo at the given 1-based index as done and return the full list",
            "parameters": {
                "type": "object",
                "properties": {
                    "index": {"type": "integer"},
                    "completion_notes": {"type": "string"},
                },
                "required": ["index", "completion_notes"],
                "additionalProperties": False,
            },
        },
    },
]

In [4]:
def handle_tool_calls(tool_calls):
    results = []
    for tc in tool_calls:
        fn = globals().get(tc.function.name)
        if fn is None:
            result = "Unknown tool"
        else:
            result = fn(**json.loads(tc.function.arguments))
        results.append({
            "role": "tool",
            "content": json.dumps(result),
            "tool_call_id": tc.id,
        })
    return results

In [5]:
def loop(messages):
    while True:
        response = client.chat.completions.create(
            model="gpt-5.2", messages=messages, tools=tools
        )
        choice = response.choices[0]
        if choice.finish_reason == "tool_calls":
            messages.append(choice.message)
            messages.extend(handle_tool_calls(choice.message.tool_calls))
        else:
            print(choice.message.content)
            break

In [6]:
system_message = """
You solve problems by first creating a todo plan, then carrying out each step.
After each step, mark it complete. Reply with the final answer only.
"""
user_message = """
A train leaves Boston at 2pm at 60mph. Another leaves New York at 3pm at 80mph toward Boston.
When do they meet? (Assume Boston-NY distance is 215 miles.)
"""

todos = []
messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_message},
]
loop(messages)

Let t be hours after 2pm when they meet. Boston train travels 60t miles. NY train starts at 3pm, i.e., 1 hour after 2pm, so it travels for (t-1) hours (requiring t>=1), covering 80(t-1) miles.
Set total distance: 60t + 80(t-1) = 215. So 60t+80t-80=215 => 140t=295 => t=295/140=59/28≈2.10714 hours after 2pm.
2.10714 hours = 2 hours + 0.10714*60 ≈ 2 hours 6.43 minutes. Meeting time ≈ 4:06 pm (about 4:06:26 pm).
Let \(t\) be the number of hours after 2pm when they meet.

- Boston train distance: \(60t\)
- New York train leaves at 3pm, so it travels \(t-1\) hours: distance \(80(t-1)\)

They meet when distances sum to 215 miles:
\[
60t + 80(t-1) = 215
\]
\[
60t + 80t - 80 = 215 \Rightarrow 140t = 295 \Rightarrow t=\frac{295}{140}=\frac{59}{28}\approx 2.1071
\]

That’s \(2.1071\) hours after 2pm \(= 2\) hours \(+ 0.1071\times 60 \approx 6.43\) minutes.

**They meet at about 4:06 pm** (approximately 4:06:26 pm).
